# 🚀 PaddleOCR v5 Vietnamese - Training on Kaggle

Complete notebook để training OCR tiếng Việt với dataset 250k samples.

## ⚙️ Kaggle Settings (BẮT BUỘC):
1. **Accelerator**: GPU T4 x2 (hoặc P100 x2)
2. **Internet**: ON (để download pretrained model)
3. **Add Data**: vietnamese-ocr-250k (dataset của bạn)

## 📊 Dataset Structure:
Dataset phải có cấu trúc:
```
vietnamese-ocr-250k/
├── images/
│   ├── img_001.jpg
│   ├── img_002.jpg
│   └── ...
└── rec_gt.txt
```

File `rec_gt.txt`:
```
img_001.jpg\tXin chào Việt Nam
img_002.jpg\tPaddleOCR v5 rất mạnh
```

## ⏱️ Timeline:
- Setup: 5-7 phút
- Data preparation: 3-5 phút
- Training: 16-20 giờ (250k samples)
- Export: 2 phút

**Tổng**: ~17-21 giờ

---

## 🎯 Bắt đầu!

## Cell 1: Clone Project (1 phút)

⚠️ **Sửa URL**: Thay `YOUR_USERNAME` bằng GitHub username của bạn!

In [ ]:
# Clone project từ GitHub
!git clone https://github.com/YOUR_USERNAME/paddleocr-v5-vietnamese.git
%cd paddleocr-v5-vietnamese

# Check files
!ls -la

## Cell 2: Setup Environment (5-7 phút)

Cài đặt PaddlePaddle, dependencies, download pretrained model.

In [ ]:
# Run setup script
!bash setup_kaggle.sh

## Cell 3: Prepare Data (3-5 phút)

⚠️ **Sửa dataset name**: Thay `/kaggle/input/vietnamese-ocr-250k` bằng tên dataset của bạn!

Script này sẽ:
- Đọc `rec_gt.txt`
- Copy images từ `images/` sang `train_data/` và `val_data/`
- Tự động split train/val (90/10)
- Tạo `train_list.txt` và `val_list.txt`
- Tạo dictionary tiếng Việt

In [ ]:
# Prepare dataset
!python prepare_data.py \
    --input_dir /kaggle/input/vietnamese-ocr-250k \
    --images_dir images \
    --label_file rec_gt.txt \
    --train_ratio 0.9

### 🔍 Verify Data

Check data đã chuẩn bị đúng chưa:

In [ ]:
# Check data structure
!echo "📁 Data directories:"
!ls -lh data/

!echo "\n📊 Training samples:"
!wc -l data/train_list.txt

!echo "\n📊 Validation samples:"
!wc -l data/val_list.txt

!echo "\n📝 Sample labels (first 5):"
!head -5 data/train_list.txt

!echo "\n🔤 Dictionary info:"
!wc -l dict/vi_dict.txt
!head -20 dict/vi_dict.txt

## Cell 4a: Test với 10k samples (30 phút) ⚡

**🎯 KHUYẾN NGHỊ: Chạy test trước!**

Test này sẽ:
- Train với 10k samples (thay vì 250k)
- Chạy 5 epochs (thay vì 100)
- Mất ~30 phút (thay vì 16-20 giờ)
- Verify setup, config, data hoạt động đúng

**Sau test xong, nếu accuracy >70% → OK để train full!**

In [ ]:
# Test training với 10k samples
!bash train_test_10k.sh

### 📊 Check Test Results

Xem accuracy sau test:

In [ ]:
# Check test results
!echo "📈 Test Accuracy:"
!tail -50 logs/test_10k_*.log | grep -i "acc:" | tail -5

!echo "\n📁 Test Model:"
!ls -lh output/test_10k/

!echo "\n💡 If accuracy >70%, proceed to full training!"

## Cell 4b: Training Full (16-20 giờ) 🚀

**⚠️ Chỉ chạy nếu test OK (accuracy >70%)**

Training full sẽ:
- Restore toàn bộ dataset (250k)
- Train 100 epochs
- Mất 16-20 giờ
- Save checkpoint mỗi 5 epochs

**Script sẽ hỏi confirm trước khi bắt đầu**

In [ ]:
# Full training với toàn bộ data
!bash train_full.sh

## Cell 5: Monitor Progress (Run trong cell riêng)

Chạy cell này trong khi training để xem progress:

In [ ]:
# Check log realtime
!tail -30 logs/train_*.log

In [ ]:
# Check GPU usage
!nvidia-smi

In [ ]:
# List checkpoints
!ls -lh output/vi_ppocr_v5/

## Cell 6: Resume Training (Nếu bị timeout)

Nếu Kaggle timeout, chạy cell này để resume:

In [ ]:
# Resume from latest checkpoint
!bash train_kaggle.sh --resume

## Cell 7: Evaluation

Sau khi training xong, evaluate model:

In [ ]:
# Evaluate best model
%cd PaddleOCR

!python tools/eval.py \
    -c ../config_kaggle.yml \
    -o Global.pretrained_model=../output/vi_ppocr_v5/best_accuracy

%cd ..

## Cell 8: Export Model (2 phút)

In [ ]:
# Export to inference format
!bash export_model.sh

## Cell 9: Test Model

Test với một ảnh validation:

In [ ]:
# Test với ảnh mẫu
!ls data/val_data/ | head -1 | xargs -I {} python test_model.py --image data/val_data/{}

## Cell 10: Package for Download

In [ ]:
# Create downloadable package
!tar -czf vi_ppocr_v5_model.tar.gz \
    inference/ \
    dict/ \
    config_kaggle.yml \
    test_model.py

!ls -lh vi_ppocr_v5_model.tar.gz

print("\n✅ Model ready for download!")
print("📥 Download từ Kaggle Output tab (bên phải)")

## Cell 11: Visualize Training (Optional)

In [ ]:
import matplotlib.pyplot as plt
import re

# Parse training log
log_files = !ls logs/train_*.log
log_file = log_files[0] if log_files else None

if log_file:
    losses = []
    accs = []
    
    with open(log_file, 'r') as f:
        for line in f:
            if 'loss:' in line:
                match = re.search(r'loss: ([\d.]+)', line)
                if match:
                    losses.append(float(match.group(1)))
            
            if 'acc:' in line:
                match = re.search(r'acc: ([\d.]+)', line)
                if match:
                    accs.append(float(match.group(1)))
    
    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    ax1.plot(losses)
    ax1.set_title('Training Loss', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Step')
    ax1.set_ylabel('Loss')
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(accs)
    ax2.set_title('Validation Accuracy', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Evaluation Step')
    ax2.set_ylabel('Accuracy')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\n📈 Statistics:")
    if losses:
        print(f"   Final Loss: {losses[-1]:.4f}")
    if accs:
        print(f"   Best Accuracy: {max(accs):.4f} ({max(accs)*100:.2f}%)")
else:
    print("No training log found")

## 🎉 Hoàn thành!

### Files đã tạo:
1. **Trained Model**: `output/vi_ppocr_v5/best_accuracy.*`
2. **Inference Model**: `inference/`
3. **Dictionary**: `dict/vi_dict.txt`
4. **Package**: `vi_ppocr_v5_model.tar.gz` (download từ Output)
5. **Training Curves**: `training_curves.png`

### Sử dụng model:
```python
from paddleocr import PaddleOCR

ocr = PaddleOCR(
    use_angle_cls=True,
    lang='vi',
    rec_model_dir='inference/',
    rec_char_dict_path='dict/vi_dict.txt'
)

result = ocr.ocr('test.jpg', cls=True)
```

### Next steps:
1. Download model về local
2. Test với ảnh thực tế
3. Deploy lên production

---

**🌟 Chúc mừng! Bạn đã training thành công PaddleOCR v5 cho tiếng Việt!**